# 步驟 3：模型訓練與 Walk-Forward 預測

運用 **LightGBM** 模型與 **Walk-Forward Purged CV** 模式避免偷看未來資料，並透過訊號濾網（Regime Filter）決定倉位大小。

## Walk-Forward Purged CV 架構

```
時間軸: 2019-01 ──────────────────────────────────────── 2026-03

第 1 次: [Train: 48M] [Purge: 1M] [Test → 3M]
第 2 次:    [Train: 48M] [Purge: 1M] [Test → 3M]
...
共約 25 次 retrain，覆蓋 74 個測試月份
```

## LightGBM 正則化參數（防 Overfitting）

| 參數 | 值 | 說明 |
|------|-----|------|
| `max_depth` | 3 | 很淺的樹，降低複雜度 |
| `num_leaves` | 15 | 限制葉節點數 |
| `reg_alpha` | 0.5 | L1 正則化 |
| `reg_lambda` | 2.0 | L2 正則化（強） |
| `min_child_samples` | 30 | 最小樣本數，避免小樣本過擬合 |
| `early_stopping` | 50 rounds | 驗證集無改善即停止 |

前置條件：已執行 `01` + `02`，變數 `X`, `y`, `all_dates` 存在記憶體中。

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import lightgbm as lgb
except ImportError:
    print('[ERROR] Install lightgbm first: pip install lightgbm'); raise

# ── 常數
TRAIN_MONTHS   = 48    # 訓練視窗（月）
PURGE_MONTHS   = 1     # purge gap，防資料洩漏
STEP_MONTHS    = 3     # 每幾個月 retrain 一次
TEST_START     = pd.Timestamp('2020-01-01')   # 資料對齊後有效起點
TOP_K          = 20    # 每月選股上限
WEIGHT_TEMP    = 5.0   # Softmax 溫度（越低越集中）
INIT_CASH      = 1_000_000
LIQUIDITY_MIN  = 30_000_000  # 30日均量門檻
FEE            = 1.425 / 1000 / 3    # 買入手續費
SLIPPAGE       = 0.001

# LightGBM 參數（強化正則化）
LGBM_PARAMS = dict(
    n_estimators     = 500,
    max_depth        = 3,        # 降低樹深度
    learning_rate    = 0.008,
    num_leaves       = 15,       # 限制葉節點
    reg_alpha        = 0.5,      # L1
    reg_lambda       = 2.0,      # L2（加強）
    min_child_samples= 30,       # 最小葉節點樣本數
    subsample        = 0.7,
    colsample_bytree = 0.7,
    random_state     = 42,
    n_jobs           = -1,
    verbose          = -1,
)

In [ ]:
# ── Walk-Forward Purged Training
test_dates = [d for d in all_dates if d >= TEST_START]
print(f'Test period: {test_dates[0].strftime("%Y-%m")} ~ {test_dates[-1].strftime("%Y-%m")} ({len(test_dates)} months)')

all_preds         = []
current_train_end = None
model             = None
retrain_count     = 0

for d in tqdm(test_dates, desc='  Walk-Forward'):
    if current_train_end is None or d >= current_train_end:
        train_start  = d - pd.DateOffset(months=TRAIN_MONTHS)
        purge_cutoff = d - pd.DateOffset(months=PURGE_MONTHS)
        idx_tr = (
            (X.index.get_level_values(0) >= train_start) &
            (X.index.get_level_values(0) <  purge_cutoff)
        )
        X_tr, y_tr = X[idx_tr], y[idx_tr]
        if len(X_tr) < 200:
            continue
        cut      = int(len(X_tr) * 0.8)
        X_t_, y_t_ = X_tr.iloc[:cut], y_tr.iloc[:cut]
        X_v_, y_v_ = X_tr.iloc[cut:], y_tr.iloc[cut:]
        model = lgb.LGBMRegressor(**LGBM_PARAMS)
        model.fit(
            X_t_, y_t_,
            eval_set=[(X_v_, y_v_)],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(period=-1)],
        )
        current_train_end = d + pd.DateOffset(months=STEP_MONTHS)
        retrain_count += 1

    X_d = X[X.index.get_level_values(0) == d]
    if model is not None and not X_d.empty:
        preds_d = model.predict(X_d)
        all_preds.append(pd.DataFrame({'y_pred': preds_d}, index=X_d.index))

final_preds = pd.concat(all_preds)
print(f'\n  {len(final_preds):,} predictions -- {retrain_count} retrain cycles')

# ── Feature Importance（最後一次訓練的模型）
fi = (pd.DataFrame({'Factor': X.columns, 'Importance': model.feature_importances_})
        .sort_values('Importance', ascending=False))
print('\nFeature Importance (last model):')
display(fi.reset_index(drop=True))

In [ ]:
# ── Regime Filter（多空濾網）
# 使用 0050 作為大盤指標，三個信號加權決定是否進場
tw50        = close['0050']
sig_60ma    = (tw50 > tw50.rolling(60).mean()).astype(float) * 0.4
sig_20ma    = (tw50 > tw50.rolling(20).mean()).astype(float) * 0.3
sig_nodip   = (tw50.pct_change(1) > -0.05).astype(float) * 0.3
regime_score = sig_60ma + sig_20ma + sig_nodip

is_bull_daily   = regime_score >= 0.4
is_bull_monthly = (
    is_bull_daily.resample('ME').mean()
    .reindex(final_preds.index.get_level_values(0).unique(), method='ffill')
    .fillna(False)
)

bull_months   = int(is_bull_monthly.sum())
total_months  = len(is_bull_monthly)
print(f'Bull months: {bull_months} / {total_months} ({bull_months/total_months:.0%})')

In [ ]:
# ── 流動性過濾 + Softmax 權重建立
def softmax_weights(scores, temp=WEIGHT_TEMP, top_k=TOP_K):
    if scores.empty:
        return pd.Series(dtype=float)
    top = scores.nlargest(top_k)
    s   = top / temp
    s  -= s.max()
    e   = np.exp(s)
    return e / e.sum()

pred_wide   = final_preds['y_pred'].unstack('stock_id')
money_mth   = resample_monthly(money_wide).rolling(30).mean()  # 30日均量（月頻）

weight_rows = {}
for d in pred_wide.index:
    if not is_bull_monthly.get(d, False):
        continue
    scores = pred_wide.loc[d].dropna()
    # 流動性過濾
    if d in money_mth.index:
        liq = money_mth.loc[d]
        liquid = liq[liq >= LIQUIDITY_MIN].index
        scores = scores.reindex(scores.index.intersection(liquid))
    weight_rows[d] = softmax_weights(scores)

weights_df = pd.DataFrame(weight_rows).T.fillna(0.0)
weights_df.index = pd.to_datetime(weights_df.index)

active_months = len(weights_df)
print(f'Active months: {active_months} / {total_months}')
print(f'Latest allocation (top 5):')
display(weights_df.iloc[-1].sort_values(ascending=False).head(10))